# 42173 Advanced Natural Language Processing
## Component: Retrieval Evaluator Example

In [1]:
import torch
import torch.nn as nn
import pandas as pd
import random
import re
import numpy as np
from transformers import T5Tokenizer, T5EncoderModel
from datasets import load_dataset

/opt/anaconda3/envs/fundamentals_of_da/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Define important variables
MODEL_NAME = "t5-small"
CHECKPOINT_PATH = "../models/t5_relevance_evaluator_pubmedqa.pt"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
# Define model
class T5RelevanceScorer(nn.Module):
    """
    T5 encoder + mean pooling + linear head -> score in [-1, 1].
    """

    def __init__(self, model_name=MODEL_NAME):
        super().__init__()
        self.encoder = T5EncoderModel.from_pretrained(model_name)
        hidden = self.encoder.config.d_model
        self.head = nn.Linear(hidden, 1)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (out * mask).sum(1) / mask.sum(1).clamp(min=1e-6)
        return torch.tanh(self.head(pooled)).squeeze(-1)

In [4]:
# Load pretrained model
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

evaluator_model = T5RelevanceScorer(MODEL_NAME).to(DEVICE)
evaluator_model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
evaluator_model.eval()

Loading weights: 100%|██████████| 51/51 [00:00<00:00, 15075.73it/s]


T5RelevanceScorer(
  (encoder): T5EncoderModel(
    (shared): Embedding(32128, 512)
    (encoder): T5Stack(
      (embed_tokens): Embedding(32128, 512)
      (block): ModuleList(
        (0): T5Block(
          (layer): ModuleList(
            (0): T5LayerSelfAttention(
              (SelfAttention): T5Attention(
                (q): Linear(in_features=512, out_features=512, bias=False)
                (k): Linear(in_features=512, out_features=512, bias=False)
                (v): Linear(in_features=512, out_features=512, bias=False)
                (o): Linear(in_features=512, out_features=512, bias=False)
                (relative_attention_bias): Embedding(32, 8)
              )
              (layer_norm): T5LayerNorm()
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (1): T5LayerFF(
              (DenseReluDense): T5DenseActDense(
                (wi): Linear(in_features=512, out_features=2048, bias=False)
                (wo): Linear(in_features=204

In [5]:
# Define function to join context
def join_context(example):
    """
    Example['context']['contexts'] is a list of abstract sections - join into one string.
    """
    ctx = example["context"]
    if isinstance(ctx, dict) and "contexts" in ctx:
        return " ".join(ctx["contexts"])
    return str(ctx)

In [6]:
# Define evaluate relevance function
@torch.no_grad()
def evaluate_relevance(question, doc_text):
    """
    Return a relevance score in [-1, 1] for a (question, document) pair.
    """
    text = f"question: {question} document: {doc_text}"
    enc = tokenizer(text, truncation=True, max_length=384, padding="max_length", return_tensors="pt")
    score = evaluator_model(enc["input_ids"].to(DEVICE), enc["attention_mask"].to(DEVICE))
    return float(score.item())

In [7]:
# Get PQA-A test set
N_TEST = 100
pubmedqa_a = load_dataset("qiaojin/PubMedQA", "pqa_artificial", split="train")

records_a = []
for ex in pubmedqa_a:
    records_a.append({
        "question": ex["question"],
        "context": join_context(ex),
        "final_decision": ex.get("final_decision", None),
    })

pubmedqa_a_df = pd.DataFrame(records_a)
test_df_a = pubmedqa_a_df.sample(n=N_TEST, random_state=42).reset_index(drop=True)
print(f"Test (from PQA-A): {len(test_df_a)}")

Test (from PQA-A): 100


In [8]:
# Evaluate relevance
test_results_a = []
all_contexts_a = pubmedqa_a_df["context"].tolist()

for _, row in test_df_a.iterrows():
    question = row["question"]
    gold_context = row["context"]
    wrong_context = random.choice([c for c in all_contexts_a if c != gold_context])

    score_gold = evaluate_relevance(question, gold_context)
    score_wrong = evaluate_relevance(question, wrong_context)

    test_results_a.append({
        "question": question[:80] + ("..." if len(question) > 80 else ""),
        "score_gold_context": round(score_gold, 3),
        "score_wrong_context": round(score_wrong, 3),
        "correctly_separated": score_gold > score_wrong,
    })

test_results_a_df = pd.DataFrame(test_results_a)
test_results_a_df

,question,score_gold_context,score_wrong_context,correctly_separated
0,Do the Norwegian national project for ethics s...,0.911,-0.930,True
1,Is heart-type fatty acid binding protein an ea...,0.919,-0.956,True
2,Does nitric oxide modulate spreading depolariz...,0.917,-0.959,True
3,Do vitamin D receptor alleles predict growth a...,0.921,-0.942,True
4,Does new FIGO staging system of vulvar cancer ...,0.910,-0.958,True
...,...,...,...,...
95,Is there no evidence for an increase in acute ...,0.911,-0.959,True
96,Do effects of activated T cells on osteoclasto...,0.922,0.718,True
97,Does aP-2β be a Downstream Effector of PITX2 R...,0.924,-0.955,True
98,"Is lipolysis -- not inflammation , cell death ...",0.925,-0.955,True


In [9]:
n_correct = test_results_a_df["correctly_separated"].sum()
print(f"Correctly separated (true-context score > wrong-context score): {n_correct}/{len(test_results_a_df)}")

Correctly separated (true-context score > wrong-context score): 100/100


In [10]:
# Define function to split context into strips
def split_into_strips(text, sentences_per_strip=2):
    """
    Split context into strips of 1-2 sentences.
    """
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    return [" ".join(sents[i:i+sentences_per_strip])
            for i in range(0, len(sents), sentences_per_strip) if sents[i:i+sentences_per_strip]]

In [11]:
# Evaluate relevance at strip level
sample_df = test_df_a.sample(n=2, random_state=42).reset_index(drop=True)
sentence_level_results = []

for _, row in sample_df.iterrows():
    question = row["question"]
    gold_context = row["context"]
    print(f"Question: {question} \n")

    # Split context into sentences
    sentences = split_into_strips(gold_context, sentences_per_strip=1)

    for sent_idx, sentence in enumerate(sentences):
        print(f"Sentences {sent_idx}: {sentence} \n")
        score = evaluate_relevance(question, sentence)
        sentence_level_results.append({
            "question": question[:60] + ("..." if len(question) > 60 else ""),
            "sentence_idx": sent_idx,
            "sentence": sentence,
            "relevance_score": round(score, 3),
            "relevant": score > 0.0,
        })

Question: Does anatomic-based three-dimensional planning preclude use of catheter-delivered contrast for treatment of prostate cancer? 

Sentences 0: Retrograde urethrography is a standard method to identify the prostatic apex during planning for prostate cancer radiotherapy. 

Sentences 1: This is an invasive and uncomfortable procedure. 

Sentences 2: With modern three-dimensional computed tomography planning, we explored whether retrograde urethrography was still necessary to accurately identify the prostatic apex. 

Sentences 3: Fifteen patients underwent computed tomography simulation with and without bladder, urethral, and rectal contrast. 

Sentences 4: The prostatic base and apex were identified on both scans, using contrast and anatomy, respectively. 

Sentences 5: The anatomic location of the prostatic apex as defined by these methods was confirmed in another 57 patients with postbrachytherapy imaging. 

Sentences 6: The prostatic base and apex were within a mean of 3.8 mm be

In [12]:
sentence_level_df = pd.DataFrame(sentence_level_results)
sentence_level_df

,question,sentence_idx,sentence,relevance_score,relevant
0,Does anatomic-based three-dimensional planning...,0,Retrograde urethrography is a standard method ...,0.130,True
1,Does anatomic-based three-dimensional planning...,1,This is an invasive and uncomfortable procedure.,-0.827,False
2,Does anatomic-based three-dimensional planning...,2,With modern three-dimensional computed tomogra...,-0.363,False
3,Does anatomic-based three-dimensional planning...,3,Fifteen patients underwent computed tomography...,-0.836,False
4,Does anatomic-based three-dimensional planning...,4,The prostatic base and apex were identified on...,-0.795,False
5,Does anatomic-based three-dimensional planning...,5,The anatomic location of the prostatic apex as...,-0.838,False
6,Does anatomic-based three-dimensional planning...,6,The prostatic base and apex were within a mean...,-0.889,False
7,Does anatomic-based three-dimensional planning...,7,"In every case, the beak of the retrograde uret...",-0.904,False
8,Does anatomic-based three-dimensional planning...,8,"With these anatomic relationships defined, in ...",-0.809,False
9,Does anatomic-based three-dimensional planning...,9,"On lateral films, we found that the urethra ex...",-0.918,False
